# CS156: Pipeline - First Draft

**Do I sound like AI, or have I always been a bad writer?**  

With the amount of AI we use on a day to day, I am fairly sure that a lot of the content I consume, whether it be the news, social media, or even the PCW that I grade, might be AI-assisted, if not completely AI-generated. This makes me wonder: as a symptom of consuming so much 'slop,' could my own writing also be showing traits of AI-generated text? 

This report answers this question by first training a model to classify assignments I wrote pre-Fall 2024, when AI tools have shown enough progress to make using it as an aid in coursework a viable strategy, from later submissions. Then, I use a (INSERT WHAT I ACTUALLY DO BASED ON PRELIMINARY MODEL RESULTS). 


## The Data

To create the dataset to be used in this pipeline, I retrieved past written assignments stored in my Minerva email's Google Drive using the notebook here ran in Colab ([GitHub](https://colab.research.google.com/drive/1f_YmZ3cR82UPcLH7uNJ9DUu-x7Gllc7c#scrollTo=e160UdxwwBmt)). Prior to the retreival, I manually labelled the documents that I wish to include. 

Importantly, since the focus of the models trained here is stylistic drift in *my* writing, I excluded the following assignments: 
- **Skill builders from CS111 and CS113:** The content is mostly mathematical equations and instructions, and they do not contain much original writing. In addition, they are hand-written documents, which adds more complexity to the ingestion process. 
- **Group assignments:** These contain writing by other students, thus were excluded to limit the training data to my own writing. 
- **Heavily templated assignments:** All assignments formatted as workbooks (both Forum codebooks and worksheets with instructions included) were excluded to prevent 'dilution' of my writing due to the instruction text included in the document. This resulted in the exclusion of several FA50 and FA51 (previously CS50 and CS51) assignments, Deep Dives from CS111 and CS113, problems sets from CS114, and some SS110 and SS152 assginments that are formatted as cover sheets. 

For the same reasons, I included any non-essay documents that reflect my own prose, such as pre-written scripts for any videos I submitted as assignments, or reflection documents that supplement technical interviews. 

Based on these exclusion criteria, 56 assignment documents qualified for inclusion in the dataset. However, 4 of these assignments were created outside of Google Drive (e.g., PDFs exported from OverLeaf), and I excluded them from the final dataset. Extracting text from PDFs would require different data processing compared to Google Drive documents, and given the small number of such files, I chose to maintain consistency by excluding them entirely.

## Loading the Data

Using (HOW DO I SCRAPE GDRIVE LOOK THIS UP), I loaded the text data from my past assignments in to a pandas dataframe and tokenized (UPDATE AS I ADD CODE). 

In [ ]:
# Install necessary packages
!pip install pandas numpy scikit-learn matplotlib seaborn scipy textstat nltk

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 33.6 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 24.1 MB/s eta 0:00:0000:0100:01
  Using cached scikit_learn-1.8.0-cp311-cp311-macosx_12_0_arm64.whl (8.1 MB)
  Using cached matplotlib-3.10.8-cp311-cp311-macosx_11_0_arm64.whl (8.1 MB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 294.9/294.9 kB 23.5 MB/s eta 0:00:00
  Using cached scipy-1.17.0-cp311-cp311-macosx_14_0_arm64.whl (20.1 MB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.1/177.1 kB 5.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 23.5 MB/s eta 0:00:00a 0:00:01
  Using cached joblib-1.5.3-py3-none-any.whl (309 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl (18 kB)
  Using cached contourpy-1.3.3-cp311-cp311-macosx_11_0_arm64.whl (270 kB)
  Using cached cycler-0.12.1-py3-none-any.whl (8.3 kB)
  Using cached fonttools-4.61.1-cp311-cp311-macosx_10_9_universal2.whl (2.9 MB)
  Using cached kiwisolve

In [ ]:
# Imports

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import re

# Sklearn imports
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.decomposition import PCA

# Stylometric analysis
import textstat
import nltk
from nltk.tokenize import word_tokenize, sent_tokenize
from collections import Counter

In [94]:
# Loading dataset

df = pd.read_csv("gdrive_scraped_data.csv", encoding='latin-1')
df.head()


file_id  \
0  1z4dvX8KX5spMNnUNkWoboNuA6doneGrNYJJWlvB2Wno   
1  19du57lm4DB4yeZuFAlyTAdgp3u-ug8MV4eLIbQNwZk8   
2  1yNA1ZfpoNLkbVhCHcuCIGBHFT1eCHsjKcmN4_icO7Yw   
3  16AVkozInKE_ol_qFQMXxQOy3OMK5kHQjDvgRUQ1cUSk   
4  1gBiAfz7LWm-IePjnEvHuelT03Uocw6l9ywwbvunsxX4   

                                        name              created_time  \
0        Reflection on Track Options [final]  2026-01-30T19:51:16.879Z   
1                                LBA [final]  2026-02-07T12:24:25.590Z   
2  Elevation Reflection & Engagement [final]  2026-01-07T08:37:17.604Z   
3            Ethnographic Assignment [final]  2026-02-10T13:05:32.687Z   
4                   Final assignment [final]  2025-12-04T22:18:34.515Z   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                        

## Pre-processing and EDA

While I am most interested in stylistic drift, I decided to first train a classifier to understand if there is any meaningful difference between my pre-AI and post-Ai writing. 

### Data Cleaning to Prevent Leakage

To ensure the classifier learns stylistic patterns rather than metadata, I create multiple versions of the text with different levels of cleaning:

**Text columns created:**
- `text_raw`: Original text with all content (baseline)
- `no_cover`: Cover sheet removed only (strips course code, instructor name, submission date)
- `text_clean`: Fully cleaned - removes cover sheet AND all formatting elements (AI statements, references, appendices, footnotes including HC tags)

**Metadata extracted:**
- `course_code`: Extracted from cover sheet for one-hot encoding

**Rationale for cleaning:**
- **Cover sheet removal**: Eliminates temporal markers (dates, instructors) and course identifiers that could leak time periods
- **Formatting removal**: Strips references, appendices, footnotes (including HC applications), and AI statements. These elements represent assignment structure rather than core writing style, and their presence/absence could leak information about course requirements or time periods

This approach allows comparison of model performance on raw text vs. main body prose to determine whether the classifier learns from content/metadata or actual writing style.

In [ ]:
# Preprocessing functions

VALID_COURSES = ['CP192', 'CS146', 'GL96', 'CS156', 'SS111',
                 'CP191', 'CS166', 'GL95', 'SS152', 'SS156', 
                 'CS113', 'CS114', 'GL94', 'SS110', 
                 'CS110', 'CS111', 'GL93', 'SS112', 
                 'CX51', 'EA51', 'FA51', 'GL92', 'MC51', 
                 'CX50', 'EA50', 'FA50', 'GL91', 'MC50', 'IL199']

def extract_course_code(text):
    """
    Extract course code from cover sheet (second line under title).
    Returns tuple: (course_code, needs_review_flag)
    """
    # Look for pattern: 2 letters + 2-3 digits
    # Search in first 500 characters (cover sheet area)
    first_section = text[:500]
    
    # Pattern: course prefix (2 letters) + course number (2-3 digits)
    pattern = r'\b([A-Z]{2})[\s-]?(\d{2,3})\b'
    matches = re.findall(pattern, first_section)
    
    if matches:
        # Reconstruct course code
        prefix, number = matches[0]
        course_code = f"{prefix}{number}"
        
        # Check if it's in our valid list
        needs_review = course_code not in VALID_COURSES
        return course_code, needs_review
    
    return None, True  # Flag for manual review if not found

def remove_cover_sheet(text):
    """
    Remove cover sheet content using layered heuristics.

    Order:
    1) Minerva-specific header detection
    2) Major page break (6+ blank lines)
    3) Early horizontal rule
    4) Course code fallback
    """
    text = text.replace('\r\n', '\n')

    if 'Minerva University' in text[:500]:
        rule_match = re.search(r'\n\s*_{5,}\s*\n', text)
        toc_match = re.search(r'(?i)\n\s*(table of )?contents\s*\n', text)
        candidates = []
        if rule_match:
            candidates.append(('rule', rule_match.start(), rule_match.end()))
        if toc_match:
            candidates.append(('toc', toc_match.start(), toc_match.end()))
        if candidates:
            kind, start, end = min(candidates, key=lambda x: x[1])
            return text[start:] if kind == 'toc' else text[end:]

    page_break_pattern = r'(\n\s*){6,}'
    match = re.search(page_break_pattern, text)
    if match:
        return text[match.end():]

    horizontal_rule_pattern = r'\n\s*_{5,}\s*\n'
    match = re.search(horizontal_rule_pattern, text[:2000])
    if match:
        return text[match.end():]

    course_code_pattern = r'\b([A-Z]{2})[\s-]?(\d{2,3})\b'
    match = re.search(course_code_pattern, text[:500])
    if match:
        pos = match.end()
        for _ in range(4):
            next_newline = text.find('\n', pos)
            if next_newline == -1:
                break
            pos = next_newline + 1
        return text[pos:]

    return text

def remove_hc_tags(text):
    """
    Remove HC tags and their applications.
    Pattern: #tag followed by : and explanation text until a double newline.
    """
    pattern = r'\#[A-Za-z0-9\-]+:[^\n]*(?:\n(?!\n)[^\n]*)*'
    return re.sub(pattern, '', text)

def remove_toc(text):
    """
    Remove table of contents blocks.
    Handles explicit headers or dense page-number lists near the top.
    """
    text = text.replace('\r\n', '\n')

    toc_pattern = r'(?i)(table of )?contents\s*\n[\s\S]*?(_{5,}|\n\n[A-Z])'
    match = re.search(toc_pattern, text)
    if match:
        return text[:match.start()] + text[match.start(2):]

    lines = text.split('\n')
    sample = lines[:20]
    toc_like = [ln for ln in sample if re.search(r'\s+\d+\s*$', ln)]
    if len(toc_like) >= 3:
        for i in range(len(sample) - 1):
            if sample[i].strip() == '' and sample[i + 1].strip() == '':
                return '\n'.join(lines[i + 1:]).lstrip()
        for i in range(len(sample)):
            if sample[i].strip() == '':
                return '\n'.join(lines[i + 1:]).lstrip()

    return text

def extract_body_only(text):
    """
    Extract main body text by stopping at Word Count or AI Statement markers.
    """
    pattern = r'(?i)(word count|ai statement|ai use)'
    match = re.search(pattern, text)
    if match:
        return text[:match.start()].strip()

    return text


def preprocess_dataframe(df):
    """
    Apply all preprocessing steps to create new text columns.

    Columns created:
    - text_raw: baseline
    - no_cover: cover sheet + TOC removed
    - text_clean: no_cover with end-matter + HC tags removed
    - course_code: extracted course code
    - label: 0 = pre-AI, 1 = post-AI
    """
    df_processed = df.copy()
    course_info = df_processed['text_raw'].apply(extract_course_code)
    df_processed['course_code'] = course_info.apply(lambda x: x[0])
    df_processed['no_cover'] = df_processed['text_raw'].apply(remove_cover_sheet).apply(remove_toc)
    df_processed['text_clean'] = df_processed['no_cover'].apply(extract_body_only).apply(remove_hc_tags)
    PRE_AI_COURSES = ['CX51', 'EA51', 'FA51', 'GL92', 'MC51', 
                      'CX50', 'EA50', 'FA50', 'GL91', 'MC50', 'IL199']
    df_processed['label'] = df_processed['course_code'].apply(
        lambda x: 0 if x in PRE_AI_COURSES else 1
    )
    
    return df_processed

# Apply preprocessing
df_processed = preprocess_dataframe(df)

# Keep only relevant columns
df_processed = df_processed[['name', 'course_code', 'label', 'text_raw', 'no_cover', 'text_clean']]

# Display summary
print("Preprocessing complete.")

print(f"\nTotal documents: {len(df_processed)}")

# Display head with truncated text columns for readability
display_df = df_processed.head().copy()
text_cols = ['text_raw', 'no_cover', 'text_clean']
for col in text_cols:
    display_df[col] = display_df[col].apply(lambda x: (x[:75] + '...') if isinstance(x, str) and len(x) > 75 else x)
display(display_df[['course_code', 'name', 'label'] + text_cols])


Preprocessing complete.

Total documents: 52


,course_code,name,label,text_raw,no_cover,text_clean
0,CP192,Reflection on Track Options [final],1,ï»¿Reflection on Track Options\r\n\r\n\r\n\r\n\r\nMinerva University\r\nCP192: Capstone...,Reflection on Track Options\nProcess Documentation\nTo ideate what kind of tr...,Reflection on Track Options\nProcess Documentation\nTo ideate what kind of tr...
1,SS111,LBA [final],1,ï»¿LBA: Analyzing the Bangle Market of Charminar\r\n\r\n\r\n\r\n\r\nMinerva Universit...,LBA: Analyzing the Bangle Market of Charminar\n Located in the âOld...,LBA: Analyzing the Bangle Market of Charminar\n Located in the âOld...
2,GL96,Elevation Reflection & Engagement [final],1,ï»¿Elevation Reflection & Engagement \r\nGL96\r\nSuisei Nakagawa\r\n\r\n\r\nOne way I...,One way I plan to engage meaningfully with Hyderabad is through its local c...,One way I plan to engage meaningfully with Hyderabad is through its local c...
3,GL96,Ethnographic Assignment [final],1,ï»¿Ethnographic Assignment\r\n\r\n\r\n\r\n\r\nMinerva University\r\nGL96: Global Learni...,"Ethnographic Assignment\nIntroduction\nFor this assignment, I spoke to Arjun ...","Ethnographic Assignment\nIntroduction\nFor this assignment, I spoke to Arjun ..."
4,SS156,Final assignment [final],1,ï»¿Tab 1\r\n\r\n\r\n\r\n\r\n\r\n\r\nComparative Analysis of Political Systems: Furusao No...,________________\nComparative Analysis of Political Systems: Furusao Nozei i...,________________\nComparative Analysis of Political Systems: Furusao Nozei i...


In [98]:
# Manual inspection of all documents

pd.set_option('display.max_colwidth', None)

for idx, row in df_processed.iterrows():
    print(f"\n{'='*100}")
    print(f"📄 Document #{idx + 1}: {row['name']}")
    print(f"   Course: {row['course_code']} | Label: {row['label']} ({'pre-AI' if row['label'] == 0 else 'post-AI'})")
    print(f"{'='*100}")
    
    # Show first 200 and last 200 chars of no_cover
    print(f"\n[NO_COVER - First 200 chars]")
    print(row['no_cover'][:200])
    print(f"\n[NO_COVER - Last 200 chars]")
    print(row['no_cover'][-200:])
    
    # Show first 200 and last 200 chars of text_clean
    print(f"\n[TEXT_CLEAN - First 200 chars]")
    print(row['text_clean'][:200] if isinstance(row['text_clean'], str) else row['text_clean'])
    print(f"\n[TEXT_CLEAN - Last 200 chars]")
    print(row['text_clean'][-200:] if isinstance(row['text_clean'], str) else row['text_clean'])
    
    print(f"\n{'='*100}\n")



📄 Document #1: Reflection on Track Options [final]
   Course: CP192 | Label: 1 (post-AI)

[NO_COVER - First 200 chars]
Reflection on Track Options
Process Documentation
To ideate what kind of track to propose, I answered the provided questions below. 
Of the tracks that would fulfill the requirements of your major(s),

[NO_COVER - Last 200 chars]
and will inform how I approach both my remaining coursework and eventual capstone proposal. 
AI Use 
I used Grammarly.com for sentence-level editing of the final document. No other AI tools were used.

[TEXT_CLEAN - First 200 chars]
Reflection on Track Options
Process Documentation
To ideate what kind of track to propose, I answered the provided questions below. 
Of the tracks that would fulfill the requirements of your major(s),

[TEXT_CLEAN - Last 200 chars]
an support collaborative, system-level learning that cannot be effectively guided through infrequent meetings, as is the case in traditional capstone advising.
HC/LO Applications for th

## Analysis

## Model Selection 

## Training 

In [ ]:
#@title also session 5 PCW (CHANGE LATER)

import matplotlib.pyplot as plt
import seaborn as sns
from sklearn import metrics, naive_bayes

labels, texts = df['v1'], df['v2']
vectorizer = text.TfidfVectorizer()
vec_texts = vectorizer.fit_transform(texts)


# We train a model
model = naive_bayes.MultinomialNB()
model.fit(vec_texts, labels)
pred_labels = model.predict(vec_texts)

confusion_mat = metrics.confusion_matrix(labels, pred_labels)
sns.heatmap(confusion_mat, square=True, annot=True, fmt='d', cbar=False)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.show()

# some checks to see how the model is doing
print(model.classes_)
print(df['v1'].value_counts())
print(vec_texts.shape) 




## Predictions

## Discussion 

## Summary 

## References 

- Session 5 PCW: https://forum.minerva.edu/app/courses/3804/sections/13026/classes/99436 